# Building Your Own Adaptive Calibration Graph

This notebook is a hands-on, build-it-yourself guide to **adaptive calibration graphs** in QUAlibrate.

By the end you will be able to build — and adapt — a complete adaptive calibration graph: it benchmarks your gates, automatically retunes the ones that fall short, and re-verifies them, all from the basic calibration nodes you already have. We cover both the single-qubit and the two-qubit (CZ) case.

We build up one concept at a time:

1. A node, and how to *copy* it with new settings
2. A basic linear graph
3. Nested sub-graphs and the orchestrator
4. Retry loops and conditional loops
5. Failure handling and the success-path sink
6. Putting it all together &rarr; the **1Q adaptive RB + retune** graph
7. The **2Q (CZ)** analog, including leakage and conditional-phase retune

Every section has a short explanation followed by a small, runnable code cell.

## 0. Prerequisites

- A generated QUAM state and an **active library** of calibration nodes (the same setup you already use to run individual nodes).
- This tutorial references nodes by the names they have in *your* library, e.g. `11_power_rabi`, `12_ramsey`, `27_single_qubit_randomized_benchmarking`. Your node names have **no `1Q_` / `cz_` prefix** &mdash; they are just the file number + name.

> If a node name used below does not exist in your library, run the "list available nodes" cell and substitute the closest match.

Building a graph (the `with QualibrationGraph.build(...)` blocks) only *constructs* the workflow and does not touch hardware. Only `graph.run()` executes on the QPU (or simulator).

In [1]:
from typing import List, Literal, Optional

from qualibrate import (
    QualibrationGraph,
    QualibrationLibrary,
    GraphParameters,
    QualibrationNode,
)
from qualibrate.core.orchestration.basic_orchestrator import BasicOrchestrator

import logging
# Keep INFO/WARNING/ERROR, suppress DEBUG noise.
logging.getLogger("qualibrate").setLevel(logging.INFO)

library = QualibrationLibrary.get_active_library()

# Make `loaded_nodes` importable no matter where the kernel started.
import sys, pathlib
for _b in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    for _c in (_b, _b / "qualibration_graphs" / "superconducting"):
        if (_c / "loaded_nodes.py").exists():
            if str(_c) not in sys.path:
                sys.path.insert(0, str(_c))
            break
    else:
        continue
    break
from loaded_nodes import loaded_nodes, load_all_nodes
load_all_nodes()  # qualibrate loads only ONE folder; widen to 1Q + CR + CZ


2026-06-24 06:42:08,005 - qm - INFO     - Starting session: 100ea2ab-a5d0-4f72-8c12-94e112b7ac18


2026-06-24 06:42:12,725 - qualibrate - WARNING - Getting calibration path from config


In [2]:
# The exact node names available to you. Use these strings in library.nodes[...].
for name in sorted(library.nodes.keys()):
    print(name)

00_close_other_qms
01_time_of_flight_mw_fem
02_resonator_spectroscopy_wide
02_resonator_spectroscopy_wide_pyloop
03_resonator_spectroscopy_single
04_twpa_calibration
05b_resonator_spectroscopy_vs_power_iq
06_resonator_spectroscopy_vs_flux
07_resonator_spectroscopy_vs_coupler_flux
08_qubit_spectroscopy
08b_qubit_spectroscopy_vs_power
08c_fullscale_dbm_adjustment
09_qubit_spectroscopy_vs_flux
09b_qubit_spectroscopy_vs_flux
10_qubit_spectroscopy_vs_coupler_flux
10b_ramsey_vs_coupler_flux
11_power_rabi
11b_rabi_chevron
12_ramsey
13_drag_calibration_180_minus_180
15a_readout_frequency_optimization
15b_readout_power_optimization
16_iq_blobs
17_xyz_delay
18_xy_coupler_delay
18a_coupler_zero_point_coarse
19a_qubit_flux_long_distortion_qubitspec
19b_qubit_flux_long_distortion_ramsey
20_qubit_flux_short_distortion
20c_leakage_error_amp
20d_cz_leakage_amplification_palea
21b_coupler_flux_long_distortion_qubitspec
21c_coupler_flux_long_distortion_ramsey
22a_coupler_flux_short_distortion
22b_all_xy

```python
library.nodes["12_ramsey"]   or   library.nodes[loaded_nodes.n12_ramsey]
```

Names start with a digit (not a valid identifier), so each alias is the name prefixed
with `n`. `loaded_nodes` reads the active library automatically, so it is always in
sync (no regeneration needed). **From here on, the cells use `loaded_nodes`.**


## 1. A node, and how to employ it

A graph is made of **nodes** taken from your library:

```python
node = library.nodes[loaded_nodes.n12_ramsey]
```

Two things you will do constantly when building graphs:

- **Override parameters** so a node runs with graph-specific settings.
- **Copy + rename** a node with `.copy(name=..., <overrides>)` so you can use the *same* underlying node more than once (for example an "initial" RB and a "verify" RB) without a name clash.

`.copy()` returns a new node instance; the original library node is untouched. Every keyword after `name=` overrides one of that node's parameters.

In [3]:
# Inspect a node and its parameters.
ramsey = library.nodes[loaded_nodes.n12_ramsey]
print("Node name:", ramsey.name)
print("Parameters:\n", ramsey.parameters)

# A renamed copy with overridden parameters (the original is untouched):
ramsey_fast = library.nodes[loaded_nodes.n12_ramsey].copy(
    name="ramsey_fast",
    num_shots=50,
)
print("\nCopied node name:", ramsey_fast.name)

2026-06-24 06:44:11,076 - qualibrate - INFO - Scanning node file D:\work\Customer_Codes\QRS\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_ramsey.py
d:\miniconda3\envs\QRS2\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-06-24 06:44:13,462 - qualibrate - INFO - Creating node 12_ramsey
2026-06-24 06:44:13,796 - qualibrate - INFO - Loaded node 12_ramsey from D:\work\Customer_Codes\QRS\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_ramsey.py
2026-06-24 06:44:13,805 - qualibrate - INFO - Creating node 12_ramsey
2026-06-24 06:44:13,915 - qualibrate - INFO - Creating node 12_ramsey
2026-06-24 06:44:13,998 - qualibrate - INFO - Copying node with name 12_ramsey with parameters name = 'ramsey_fast', node_parameters = {'n

Node name: 12_ramsey
Parameters:
 multiplexed=False use_state_discrimination=False reset_type='thermal' qubits=None num_shots=100 frequency_detuning_in_mhz=1.0 min_wait_time_in_ns=16 max_wait_time_in_ns=30000 wait_time_num_points=500 log_or_linear_sweep='log' simulate=False simulation_duration_ns=50000 use_waveform_report=True timeout=120 load_data_id=None

Copied node name: ramsey_fast


## 2. Building block 1 &mdash; a basic linear graph

The smallest useful graph: a few nodes run in sequence. The pattern is always:

1. Define a `GraphParameters` subclass (what the graph as a whole accepts).
2. Open `QualibrationGraph.build(name, parameters=...)` as a context manager.
3. `add_node(...)` for each node, and `connect(a, b)` to make `b` run after `a`.

`connect(a, b)` means *"after `a` succeeds, run `b`"*.

In [4]:
class TuneupParameters(GraphParameters):
    qubits: Optional[List[str]] = None  # None = all active qubits

with QualibrationGraph.build("basic_tuneup", parameters=TuneupParameters()) as basic_graph:
    rabi = library.nodes[loaded_nodes.n11_power_rabi]
    basic_graph.add_node(rabi)

    ramsey = library.nodes[loaded_nodes.n12_ramsey]
    basic_graph.add_node(ramsey)
    basic_graph.connect(rabi, ramsey)  # ramsey runs after rabi

    rb = library.nodes[loaded_nodes.n27_single_qubit_randomized_benchmarking]
    basic_graph.add_node(rb)
    basic_graph.connect(ramsey, rb)

# To execute on hardware/simulator:
#   basic_graph.run(qubits=["q1", "q2"])

2026-06-24 07:13:12,033 - qualibrate - INFO - Scanning node file D:\work\Customer_Codes\QRS\qualibration_graphs\superconducting\calibrations\1Q_calibrations\11_power_rabi.py
2026-06-24 07:13:12,129 - qualibrate - INFO - Creating node 11_power_rabi
2026-06-24 07:13:12,176 - qualibrate - INFO - Loaded node 11_power_rabi from D:\work\Customer_Codes\QRS\qualibration_graphs\superconducting\calibrations\1Q_calibrations\11_power_rabi.py
2026-06-24 07:13:12,179 - qualibrate - INFO - Creating node 11_power_rabi
2026-06-24 07:13:12,239 - qualibrate - INFO - Creating node 12_ramsey
2026-06-24 07:13:12,314 - qualibrate - INFO - Scanning node file D:\work\Customer_Codes\QRS\qualibration_graphs\superconducting\calibrations\1Q_calibrations\27_single_qubit_randomized_benchmarking.py
2026-06-24 07:13:12,495 - qualibrate - INFO - Creating node 27_single_qubit_randomized_benchmarking
2026-06-24 07:13:12,562 - qualibrate - INFO - Loaded node 27_single_qubit_randomized_benchmarking from D:\work\Customer_Co

## 3. Building block 2 &mdash; nested sub-graphs and the orchestrator

You can nest a graph inside a graph. A **sub-graph** is built exactly like a graph, then added to the parent with `add_node(subgraph)` and wired with `connect(...)`. This lets you package a reusable block (e.g. a "pi-pulse tune-up") and drop it into larger workflows.

When you build a (sub-)graph you can pass an **orchestrator**, which decides how targets traverse the nodes. `BasicOrchestrator(skip_failed=False)` means: *run every node for every target, even if an upstream node failed for that target.* This is the convention we want for a retune, because every corrective step should run.

In [ ]:
with QualibrationGraph.build("nested_demo", parameters=TuneupParameters()) as parent_graph:
    rabi = library.nodes[loaded_nodes.n11_power_rabi]
    parent_graph.add_node(rabi)

    # A reusable sub-graph, built with its own orchestrator.
    with QualibrationGraph.build(
        "pi_pulse_tuneup",
        parameters=TuneupParameters(),
        orchestrator=BasicOrchestrator(skip_failed=False),
    ) as subgraph:
        ramsey = library.nodes[loaded_nodes.n12_ramsey]
        subgraph.add_node(ramsey)
        drag = library.nodes[loaded_nodes.n13_drag_calibration_180_minus_180]
        subgraph.add_node(drag)
        subgraph.connect(ramsey, drag)

    parent_graph.add_node(subgraph)
    parent_graph.connect(rabi, subgraph)  # the whole sub-graph runs after rabi

## 4. Building block 3 &mdash; retry loops

`graph.loop(node, max_iterations=N)` runs a node up to `N` times. Use it to simply repeat a measurement (e.g. to average down, or because a node is occasionally flaky).

In [ ]:
with QualibrationGraph.build("retry_demo", parameters=TuneupParameters()) as retry_graph:
    rb = library.nodes[loaded_nodes.n27_single_qubit_randomized_benchmarking]
    retry_graph.add_node(rb)
    retry_graph.loop(rb, max_iterations=3)  # run rb up to 3 times

## 5. Building block 4 &mdash; conditional loops

Most of the time you don't want a fixed number of retries &mdash; you want to retry **until a target metric is reached**. Pass a predicate via `on=`:

```python
graph.loop(node, on=predicate, max_iterations=N)
```

The predicate has the signature `predicate(node, target) -> bool` and returns `True` to keep looping for that `target`. It inspects the node's results, which calibration nodes expose as `node.results["fit_results"][target]`.

`max_iterations` is still respected as a hard cap.

In [ ]:
class AdaptiveRamseyParameters(GraphParameters):
    qubits: Optional[List[str]] = None
    target_t2_star_us: float = 10.0

with QualibrationGraph.build("adaptive_ramsey", parameters=AdaptiveRamseyParameters()) as ramsey_graph:
    ramsey = library.nodes[loaded_nodes.n12_ramsey]
    ramsey_graph.add_node(ramsey)

    def should_repeat_ramsey(node: QualibrationNode, target: str) -> bool:
        """Keep retrying until T2* exceeds the target threshold."""
        fit = node.results.get("fit_results", {}).get(target, {})
        decay = fit.get("decay")  # T2* in seconds
        if not fit.get("success") or decay is None:
            return True
        return (decay * 1e6) < ramsey_graph.parameters.target_t2_star_us

    ramsey_graph.loop(ramsey, on=should_repeat_ramsey, max_iterations=3)

## 6. Building block 5 &mdash; failure handling and the success-path sink

Nodes set an **outcome** per target: `"successful"` or `"failed"`. You can route on these:

- `connect(a, b)` &mdash; runs `b` after `a` **succeeds**.
- `connect_on_failure(a, b)` &mdash; runs `b` after `a` **fails**.

There is one validation rule to remember: **every node that has outgoing edges must have at least one success-path connection.** So if a node only routes failures somewhere, give it a success target too. A clean way to satisfy this is a tiny no-op node, `28_rb_success_exit`, used as the success sink: targets that pass simply land there and exit.

In [ ]:
with QualibrationGraph.build("failure_demo", parameters=TuneupParameters()) as fail_graph:
    rb = library.nodes[loaded_nodes.n27_single_qubit_randomized_benchmarking]
    fail_graph.add_node(rb)

    # Success path: a no-op sink so the success-connection rule is satisfied.
    success_exit = library.nodes[loaded_nodes.n28_rb_success_exit].copy(name="rb_success_exit")
    fail_graph.add_node(success_exit)
    fail_graph.connect(rb, success_exit)

    # Failure path: send failing qubits somewhere to recover.
    recover = library.nodes[loaded_nodes.n12_ramsey].copy(name="recover_ramsey")
    fail_graph.add_node(recover)
    fail_graph.connect_on_failure(rb, recover)

## 7. How a node decides "failed": `fidelity_threshold`

How does `27_single_qubit_randomized_benchmarking` know a qubit "failed"? You give it a `fidelity_threshold`. Inside the node, after fitting, any qubit whose gate fidelity is below the threshold is marked `outcomes["failed"]` &mdash; which is exactly what `connect_on_failure` routes on. The two-qubit interleaved CZ RB node (`37b_two_qubit_interleaved_cz_rb`) works the same way for CZ fidelity.

So the adaptive pattern is:

> **RB with a `fidelity_threshold` &rarr; failing targets routed into a retune sub-graph &rarr; re-verify &rarr; loop.**

## 8. Looping a *sub-graph*

In section 5 the loop predicate received a **node**. When you loop a whole **sub-graph**, the predicate instead receives the **sub-graph**, and you reach into the specific node whose results you care about:

```python
def should_keep_retuning(subgraph, target) -> bool:
    verify_node = subgraph._elements["rb_verify"]   # node by its name=
    fit = verify_node.results.get("fit_results", {}).get(target, {})
    ...
```

`subgraph._elements[name]` fetches a node inside the sub-graph by the `name=` you gave it. (It is an internal accessor, but it is the same mechanism the orchestrator uses; there is no public API for this yet.) The verification node must be the **last** node of the sub-graph so its freshly-computed metric drives the decision.

## 9. Full assembly &mdash; the 1Q adaptive RB + retune graph

Now we combine everything into one complete, self-contained adaptive RB + retune graph:

- `rb_initial` runs RB with a `fidelity_threshold`.
- Passing qubits &rarr; `rb_success_exit` (sink).
- Failing qubits &rarr; `retune_low_fidelity` sub-graph: re-find the operating point (flux + frequency), refine the pulse amplitudes, then re-verify with RB.
- The sub-graph is looped on `should_keep_retuning` until fidelity is met (or `max_retune_iterations` is hit).

First, the parameters and the loop predicate.

In [ ]:
class Parameters(GraphParameters):
    qubits: Optional[List[str]] = None          # None = all active qubits
    fidelity_threshold: float = 0.9995          # 1Q gate-fidelity acceptance
    max_retune_iterations: int = 2              # hard cap on retune cycles

parameters = Parameters()

class RetuneParameters(GraphParameters):
    # Filled in automatically with the targets routed from the parent.
    qubits: Optional[List[str]] = None

def should_keep_retuning(subgraph: QualibrationGraph, target: str) -> bool:
    """True while this qubit's verified fidelity is still below threshold."""
    rb_verify_node = subgraph._elements["rb_verify"]
    fit = rb_verify_node.results.get("fit_results", {}).get(target, {})
    if not fit or not fit.get("success"):
        return True  # no/failed fit -> keep retrying
    fidelity = 1.0 - float(fit.get("error_per_gate", 1.0))
    return fidelity < parameters.fidelity_threshold

Now the graph itself. Read it top-to-bottom: initial RB, the retune sub-graph, the success sink, the failure routing, and finally the loop.

In [ ]:
with QualibrationGraph.build("adaptive_rb_with_retune", parameters=parameters) as graph:
    # 1) Initial RB on all selected qubits, with a fidelity threshold.
    rb_initial = library.nodes[loaded_nodes.n27_single_qubit_randomized_benchmarking].copy(
        name="rb_initial",
        use_state_discrimination=True,
        num_random_sequences=100,
        max_circuit_depth=1024,
        delta_clifford=100,
        num_shots=20,
        log_scale=True,
        fidelity_threshold=parameters.fidelity_threshold,
    )
    graph.add_node(rb_initial)

    # 2) Retune sub-graph: re-find operating point -> refine amplitudes -> verify.
    with QualibrationGraph.build(
        "retune_low_fidelity",
        parameters=RetuneParameters(),
        orchestrator=BasicOrchestrator(skip_failed=False),
    ) as retune_subgraph:
        ramsey_vs_flux_calibration = library.nodes[loaded_nodes.n23_ramsey_vs_flux_calibration].copy(
            name="ramsey_vs_flux_calibration",
            num_shots=100,
            frequency_detuning_in_mhz=5.0,
            min_wait_time_in_ns=500,
            max_wait_time_in_ns=1000,
            wait_time_step_in_ns=4,
            flux_span=0.1,
            flux_num=51,
        )
        ramsey = library.nodes[loaded_nodes.n12_ramsey].copy(
            name="ramsey",
            num_shots=100,
            frequency_detuning_in_mhz=0.1,
            min_wait_time_in_ns=16,
            max_wait_time_in_ns=100_000,
            wait_time_num_points=200,
            use_state_discrimination=True,
            log_or_linear_sweep="linear",
        )
        erramp_x180 = library.nodes[loaded_nodes.n11_power_rabi].copy(
            name="power_rabi_error_amplification_x180",
            max_number_pulses_per_sweep=200,
            min_amp_factor=0.985,
            max_amp_factor=1.015,
            amp_factor_step=0.001,
            use_state_discrimination=True,
            num_shots=10,
        )
        erramp_x90 = library.nodes[loaded_nodes.n11_power_rabi].copy(
            name="power_rabi_error_amplification_x90",
            max_number_pulses_per_sweep=200,
            min_amp_factor=0.985,
            max_amp_factor=1.015,
            amp_factor_step=0.001,
            operation="x90",
            update_x90=False,
            use_state_discrimination=True,
            num_shots=10,
        )
        # The verification RB MUST be the terminal node: its fit_results drive the loop.
        rb_verify = library.nodes[loaded_nodes.n27_single_qubit_randomized_benchmarking].copy(
            name="rb_verify",
            use_state_discrimination=True,
            num_random_sequences=30,
            max_circuit_depth=1024,
            delta_clifford=100,
            num_shots=20,
            log_scale=True,
            fidelity_threshold=parameters.fidelity_threshold,
        )

        retune_subgraph.add_node(ramsey_vs_flux_calibration)
        retune_subgraph.add_node(ramsey)
        retune_subgraph.add_node(erramp_x180)
        retune_subgraph.add_node(erramp_x90)
        retune_subgraph.add_node(rb_verify)

        retune_subgraph.connect(ramsey_vs_flux_calibration, ramsey)
        retune_subgraph.connect(ramsey, erramp_x180)
        retune_subgraph.connect(erramp_x180, erramp_x90)
        retune_subgraph.connect(erramp_x90, rb_verify)

    graph.add_node(retune_subgraph)

    # 3) Success sink (satisfies the success-connection rule).
    success_exit = library.nodes[loaded_nodes.n28_rb_success_exit].copy(name="rb_success_exit")
    graph.add_node(success_exit)
    graph.connect(rb_initial, success_exit)

    # 4) Route ONLY failing qubits into the retune sub-graph.
    graph.connect_on_failure(rb_initial, retune_subgraph)

    # 5) Loop the retune until fidelity is met (capped by max_retune_iterations).
    graph.loop(
        retune_subgraph,
        on=should_keep_retuning,
        max_iterations=parameters.max_retune_iterations,
    )

In [ ]:
# Run on hardware/simulator. (In a notebook __name__ == "__main__" is True,
# so executing this cell will start a real run; comment it out to only build.)
if __name__ == "__main__":
    graph.run()

## 10. The 2Q (CZ) analog

The exact same building blocks apply to two-qubit gates. What changes:

| 1Q | 2Q (CZ) |
|----|---------|
| targets are qubits (`qubits`) | targets are qubit pairs (`qubit_pairs`, via `targets_name = "qubit_pairs"`) |
| one RB node | a **pair**: standard RB &rarr; interleaved CZ RB |
| fidelity from `1 - error_per_gate` | CZ fidelity straight from the interleaved RB (`fit_results[pair]["fidelity"]`) |
| retune re-finds flux/frequency | retune fixes **leakage** and **conditional phase** |

For the CZ retune we make two deliberate, textbook choices:

- We **include** `20d_cz_leakage_amplification_palea` at the front of the retune &mdash; a leakage-recovery step, since CZ fidelity is often limited by leakage.
- We **omit** the coarse `35_cz_phase_compensation`. A retune refines an *already-calibrated* gate (it does not start from scratch), so we go straight to the error-amplification node `35a_cz_phase_compensation_error_amp` instead of redoing coarse compensation.

In [ ]:
class CZParameters(GraphParameters):
    targets_name = "qubit_pairs"                # targets are pairs, not single qubits
    qubit_pairs: Optional[List[str]] = None     # None = all active qubit pairs
    operation: Literal[
        "cz_flattop", "cz_unipolar", "cz_bipolar", "cz_flattop_erf", "cz_SNZ"
    ] = "cz_SNZ"
    fidelity_threshold: float = 0.98            # CZ-gate-fidelity acceptance
    max_retune_iterations: int = 2

cz_parameters = CZParameters()

class CZRetuneParameters(GraphParameters):
    targets_name = "qubit_pairs"
    qubit_pairs: Optional[List[str]] = None

def should_keep_retuning_cz(subgraph: QualibrationGraph, target: str) -> bool:
    """True while this pair's CZ fidelity is still below threshold."""
    verify_node = subgraph._elements["interleaved_rb_verify"]
    fit = verify_node.results.get("fit_results", {}).get(target, {})
    if not fit or not fit.get("success"):
        return True
    fidelity = float(fit.get("fidelity", 0.0))
    return fidelity < cz_parameters.fidelity_threshold

In [ ]:
with QualibrationGraph.build("adaptive_cz_rb_with_retune", parameters=cz_parameters) as cz_graph:
    # 1) Initial RB pair: standard RB feeds StandardRB_alpha to the interleaved CZ RB.
    standard_rb_initial = library.nodes[loaded_nodes.n37_two_qubit_standard_rb].copy(
        name="standard_rb_initial",
        num_shots=200,
        use_state_discrimination=True,
        circuit_lengths=[1, 4, 16, 32, 64],
        num_circuits_per_length=5,
        seed=0,
        use_input_stream=False,
        reset_type="active",
        operation=cz_parameters.operation,
    )
    cz_graph.add_node(standard_rb_initial)

    interleaved_rb_initial = library.nodes[loaded_nodes.n37b_two_qubit_interleaved_cz_rb].copy(
        name="interleaved_rb_initial",
        num_shots=200,
        use_state_discrimination=True,
        circuit_lengths=[1, 4, 16, 32, 64],
        num_circuits_per_length=5,
        seed=0,
        use_input_stream=False,
        reset_type="active",
        operation=cz_parameters.operation,
        fidelity_threshold=cz_parameters.fidelity_threshold,
    )
    cz_graph.add_node(interleaved_rb_initial)
    cz_graph.connect(standard_rb_initial, interleaved_rb_initial)

    # 2) Retune sub-graph: leakage -> conditional-phase err-amp -> phase-comp err-amp -> verify pair.
    #    (Coarse 35_cz_phase_compensation is intentionally omitted: this is a retune.)
    with QualibrationGraph.build(
        "retune_low_cz_fidelity",
        parameters=CZRetuneParameters(),
        orchestrator=BasicOrchestrator(skip_failed=False),
    ) as cz_retune_subgraph:
        leakage_amplification = library.nodes[loaded_nodes.n20d_cz_leakage_amplification_palea].copy(
            name="leakage_amplification",
            num_shots=100,
            amp_range=0.05,
            amp_step=0.001,
            number_of_operations=20,
            use_state_discrimination=True,
            reset_type="active",
            operation=cz_parameters.operation,
        )
        conditional_phase_error_amp = library.nodes[loaded_nodes.n33_cz_conditional_phase_error_amp].copy(
            name="conditional_phase_error_amp",
            num_shots=100,
            amp_range=0.005,
            amp_step=0.0002,
            num_frame_rotations=10,
            number_of_operations=20,
            use_state_discrimination=True,
            reset_type="active",
            operation=cz_parameters.operation,
        )
        phase_compensation_error_amp = library.nodes[loaded_nodes.n35a_cz_phase_compensation_error_amp].copy(
            name="phase_compensation_error_amp",
            num_shots=100,
            num_frames=201,
            number_of_operations=40,
            use_state_discrimination=True,
            reset_type="active",
            operation=cz_parameters.operation,
        )
        # The verification RB pair MUST end the sub-graph: the interleaved RB drives the loop.
        standard_rb_verify = library.nodes[loaded_nodes.n37_two_qubit_standard_rb].copy(
            name="standard_rb_verify",
            num_shots=200,
            use_state_discrimination=True,
            circuit_lengths=[1, 4, 16, 32, 64],
            num_circuits_per_length=5,
            seed=0,
            use_input_stream=False,
            reset_type="active",
            operation=cz_parameters.operation,
        )
        interleaved_rb_verify = library.nodes[loaded_nodes.n37b_two_qubit_interleaved_cz_rb].copy(
            name="interleaved_rb_verify",
            num_shots=200,
            use_state_discrimination=True,
            circuit_lengths=[1, 4, 16, 32, 64],
            num_circuits_per_length=5,
            seed=0,
            use_input_stream=False,
            reset_type="active",
            operation=cz_parameters.operation,
            fidelity_threshold=cz_parameters.fidelity_threshold,
        )

        cz_retune_subgraph.add_node(leakage_amplification)
        cz_retune_subgraph.add_node(conditional_phase_error_amp)
        cz_retune_subgraph.add_node(phase_compensation_error_amp)
        cz_retune_subgraph.add_node(standard_rb_verify)
        cz_retune_subgraph.add_node(interleaved_rb_verify)

        cz_retune_subgraph.connect(leakage_amplification, conditional_phase_error_amp)
        cz_retune_subgraph.connect(conditional_phase_error_amp, phase_compensation_error_amp)
        cz_retune_subgraph.connect(phase_compensation_error_amp, standard_rb_verify)
        cz_retune_subgraph.connect(standard_rb_verify, interleaved_rb_verify)

    cz_graph.add_node(cz_retune_subgraph)

    # 3) Success sink for pairs that already pass.
    cz_success_exit = library.nodes[loaded_nodes.n38_cz_rb_success_exit].copy(name="cz_rb_success_exit")
    cz_graph.add_node(cz_success_exit)
    cz_graph.connect(interleaved_rb_initial, cz_success_exit)

    # 4) Route ONLY low-fidelity pairs into the retune.
    cz_graph.connect_on_failure(interleaved_rb_initial, cz_retune_subgraph)

    # 5) Loop until CZ fidelity is met (capped by max_retune_iterations).
    cz_graph.loop(
        cz_retune_subgraph,
        on=should_keep_retuning_cz,
        max_iterations=cz_parameters.max_retune_iterations,
    )

In [ ]:
# Run on hardware/simulator (executes when the cell is run; comment out to only build).
if __name__ == "__main__":
    cz_graph.run()

## 11. Make it your own

You now have the full toolkit. To build *your* adaptive graph:

- **Swap the verification metric.** Any node that writes `node.results["fit_results"][target]` and accepts a `fidelity_threshold` can drive the loop &mdash; e.g. T1, echo, or `24_zz_off_jazz` for ZZ.
- **Change the retune chain.** Add or remove corrective nodes inside the sub-graph; just keep the verification node **last** and update the `_elements["..."]` name in your predicate to match it.
- **Tune the thresholds and caps** via the graph `Parameters` (`fidelity_threshold`, `max_retune_iterations`).
- **Pick your targets.** `qubits=None` / `qubit_pairs=None` means "all active"; pass an explicit list to restrict.

### 1Q vs 2Q at a glance

| | 1Q | 2Q (CZ) |
|---|----|---------|
| targets | `qubits` | `qubit_pairs` (`targets_name = "qubit_pairs"`) |
| benchmark | `27_single_qubit_randomized_benchmarking` | `37_two_qubit_standard_rb` &rarr; `37b_two_qubit_interleaved_cz_rb` |
| loop metric | `1 - error_per_gate` | interleaved `fidelity` |
| retune focus | flux / frequency / amplitude | leakage / conditional phase |
| success sink | `28_rb_success_exit` | `38_cz_rb_success_exit` |

Everything you need is in this notebook: the two graphs above are complete and self-contained. Copy a cell, swap in your nodes and thresholds, and you have your own adaptive calibration graph.